# GCN Example 2 — Graph Classification (synthetic graphs)

The first example (`gcn_karate_club.ipynb`) labelled **individual nodes**. This notebook
tackles the second canonical graph task: labelling an **entire graph**.

## The task

We generate a synthetic dataset of small random graphs of two kinds and ask the GCN to
tell them apart:

- **Class 0 — sparse** graphs (each possible edge present with probability `p = 0.15`).
- **Class 1 — dense** graphs (`p = 0.40`).

Every graph has the same number of nodes (`N = 20`) so we can batch them as tensors.

## Node classification vs. graph classification

A node-classification GCN outputs one prediction **per node**. For graph classification
we need a single prediction **per graph**, so we add a **readout** (a.k.a. pooling) step
that collapses all the node embeddings into one fixed-size graph embedding:

```
node features --[GCN layers]--> node embeddings --[readout]--> graph embedding --[MLP]--> class
```

Here the readout concatenates the **mean** and the **max** over the nodes — mean captures
the average signal, max captures the strongest. This pattern (message passing → readout →
classifier) is exactly how GNNs classify molecules, scenes, or programs.

> Self-contained and fast on CPU — no dataset download. Tested with `tensorflow==2.1`.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

np.random.seed(0)
tf.random.set_seed(0)
print('TensorFlow', tf.__version__)

## 1. Generate the synthetic dataset

Each graph is an Erdős–Rényi random graph: we flip a biased coin for every possible edge.
A low edge probability gives a sparse graph (class 0), a high one gives a dense graph
(class 1).

For each graph we build:

- **`A_hat`** — the normalized adjacency (self-loops + symmetric degree normalization,
  exactly as in example 1).
- **`X`** — a node feature matrix. We use each node's **degree** (how many neighbours it
  has) as its single feature, which is a simple, graph-agnostic structural signal.

We stack everything into batched arrays of shape `(num_graphs, N, ...)`.

In [ ]:
N = 20            # nodes per graph
N_PER_CLASS = 250 # graphs per class

def random_graph(p):
    """Symmetric Erdos-Renyi adjacency with edge probability p (no self-loops)."""
    upper = (np.random.rand(N, N) < p).astype('float32')
    upper = np.triu(upper, k=1)
    return upper + upper.T

def normalize(A):
    A_tilde = A + np.eye(N, dtype='float32')
    d = A_tilde.sum(1)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d))
    return (d_inv_sqrt @ A_tilde @ d_inv_sqrt).astype('float32')

X_list, Ahat_list, y_list = [], [], []
for label, p in [(0, 0.15), (1, 0.40)]:
    for _ in range(N_PER_CLASS):
        A = random_graph(p)
        deg = A.sum(1, keepdims=True).astype('float32')  # degree as node feature
        X_list.append(deg / N)                           # scale to ~[0, 1]
        Ahat_list.append(normalize(A))
        y_list.append(label)

X = np.stack(X_list)          # (num, N, 1)
Ahat = np.stack(Ahat_list)    # (num, N, N)
y = np.array(y_list)          # (num,)

# Shuffle and split 80/20
perm = np.random.permutation(len(y))
X, Ahat, y = X[perm], Ahat[perm], y[perm]
split = int(0.8 * len(y))
Xtr, Xte = X[:split], X[split:]
Atr, Ate = Ahat[:split], Ahat[split:]
ytr, yte = y[:split], y[split:]

print('Dataset:', X.shape, '| train:', len(ytr), '| test:', len(yte))

## 2. The model

Two GCN layers followed by a mean+max readout and a small classifier. A handy trick: a
Keras `Dense` layer applied to a `(batch, N, features)` tensor transforms only the **last**
axis, giving us the `X W` part for free. We then `tf.matmul(A_hat, ...)` to mix
neighbours — the same `Â (X W)` operation as example 1, just **batched** over many graphs
at once.

In [ ]:
class GraphGCN(tf.keras.Model):
    def __init__(self, hidden=32, classes=2):
        super().__init__()
        self.d1 = layers.Dense(hidden, use_bias=False)   # X W  (layer 1)
        self.d2 = layers.Dense(hidden, use_bias=False)   # X W  (layer 2)
        self.classifier = tf.keras.Sequential([
            layers.Dense(hidden, activation='relu'),
            layers.Dense(classes),
        ])

    def call(self, inputs):
        X, A_hat = inputs
        h = tf.nn.relu(tf.matmul(A_hat, self.d1(X)))     # GCN layer 1
        h = tf.nn.relu(tf.matmul(A_hat, self.d2(h)))     # GCN layer 2
        # Readout: pool node embeddings into one vector per graph
        graph_emb = tf.concat([tf.reduce_mean(h, axis=1),
                               tf.reduce_max(h, axis=1)], axis=1)
        return self.classifier(graph_emb)

model = GraphGCN()
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam(0.01)

## 3. Train

The dataset is tiny, so we use **full-batch** gradient descent — one update per epoch over
all training graphs at once. We track test accuracy as we go.

In [ ]:
Xtr_t, Atr_t = tf.constant(Xtr), tf.constant(Atr)
Xte_t, Ate_t = tf.constant(Xte), tf.constant(Ate)
ytr_t = tf.constant(ytr)

EPOCHS = 100
for epoch in range(1, EPOCHS + 1):
    with tf.GradientTape() as tape:
        logits = model((Xtr_t, Atr_t))
        loss = loss_fn(ytr_t, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    if epoch % 10 == 0 or epoch == 1:
        train_acc = (tf.argmax(logits, 1).numpy() == ytr).mean()
        test_pred = tf.argmax(model((Xte_t, Ate_t)), 1).numpy()
        test_acc = (test_pred == yte).mean()
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={float(loss):.4f}  '
              f'train_acc={train_acc:.3f}  test_acc={test_acc:.3f}')

## 4. Evaluate

Final test accuracy plus a confusion-matrix-style breakdown. The model should comfortably
separate sparse from dense graphs.

In [ ]:
pred = tf.argmax(model((Xte_t, Ate_t)), 1).numpy()
acc = (pred == yte).mean()
print(f'Test accuracy: {acc:.1%}\n')

cm = np.zeros((2, 2), dtype=int)
for t, p in zip(yte, pred):
    cm[t, p] += 1
print('Confusion matrix (rows=true, cols=pred):')
print('              pred sparse  pred dense')
print(f'true sparse   {cm[0,0]:10d}  {cm[0,1]:10d}')
print(f'true dense    {cm[1,0]:10d}  {cm[1,1]:10d}')

### Things to try

- Make the task harder by moving the two probabilities closer (e.g. `0.25` vs `0.30`).
- Replace mean+max readout with **sum** pooling, or just mean — does it still separate?
- Swap the Erdős–Rényi generator for a harder, *structure*-based contrast: e.g. **two
  communities** (a stochastic block model) vs. a uniform random graph with the **same**
  number of edges. Now density is identical and the model must use structure — exactly
  where message passing earns its keep.
- Allow variable-size graphs and switch to block-diagonal batching.